from google.colab import drive
drive.mount('/content/drive')

In [1]:
import pandas as pd
import os
#os.chdir('/content/drive/MyDrive/name/IMP-OIC-Windowing')
from utils.extractframes import FrameExtractor
import graphene
directory_path = 'STAR_eval/Charades'
from gpt_ask import run_gpt
main_ds = pd.read_json('STAR_eval/STAR_val.json')
main = main_ds.loc[:, ['question_id','question','video_id','start','end','answer', 'choices']]


In [2]:
CLASSES = ['airplane', 'animal', 'arm', 'bag', 'banana', 'basket', 'beach', 'bear', 'bed', 'bench', 'bike',
                'bird', 'board', 'boat', 'book', 'boot', 'bottle', 'bowl', 'box', 'boy', 'branch', 'building',
                'bus', 'cabinet', 'cap', 'car', 'cat', 'chair', 'child', 'clock', 'coat', 'counter', 'cow', 'cup',
                'curtain', 'desk', 'dog', 'door', 'drawer', 'ear', 'elephant', 'engine', 'eye', 'face', 'fence',
                'finger', 'flag', 'flower', 'food', 'fork', 'fruit', 'giraffe', 'girl', 'glass', 'glove', 'guy',
                'hair', 'hand', 'handle', 'hat', 'head', 'helmet', 'hill', 'horse', 'house', 'jacket', 'jean',
                'kid', 'kite', 'lady', 'lamp', 'laptop', 'leaf', 'leg', 'letter', 'light', 'logo', 'man', 'men',
                'motorcycle', 'mountain', 'mouth', 'neck', 'nose', 'number', 'orange', 'pant', 'paper', 'paw',
                'people', 'person', 'phone', 'pillow', 'pizza', 'plane', 'plant', 'plate', 'player', 'pole', 'post',
                'pot', 'racket', 'railing', 'rock', 'roof', 'room', 'screen', 'seat', 'sheep', 'shelf', 'shirt',
                'shoe', 'short', 'sidewalk', 'sign', 'sink', 'skateboard', 'ski', 'skier', 'sneaker', 'snow',
                'sock', 'stand', 'street', 'surfboard', 'table', 'tail', 'tie', 'tile', 'tire', 'toilet', 'towel',
                'tower', 'track', 'train', 'tree', 'truck', 'trunk', 'umbrella', 'vase', 'vegetable', 'vehicle',
                'wave', 'wheel', 'window', 'windshield', 'wing', 'wire', 'woman', 'zebra']

REL_CLASSES = ['above', 'across', 'against', 'along', 'and', 'at', 'attached to', 'behind',
                'belonging to', 'between', 'carrying', 'covered in', 'covering', 'eating', 'flying in', 'for',
                'from', 'growing on', 'hanging from', 'has', 'holding', 'in', 'in front of', 'laying on',
                'looking at', 'lying on', 'made of', 'mounted on', 'near', 'of', 'on', 'on back of', 'over',
                'painted on', 'parked on', 'part of', 'playing', 'riding', 'says', 'sitting on', 'standing on',
                'to', 'under', 'using', 'walking in', 'walking on', 'watching', 'wearing', 'wears', 'with']

words = set(CLASSES+REL_CLASSES)

In [3]:
import string
index_list = []
for q in main.loc[:,'question_id'].values:
    questions = main.query("question_id=='"+q+"'")["question"]
    answer = main.query("question_id=='"+q+"'")["answer"].values[0].lower().strip('the').translate(str.maketrans('', '', string.punctuation))
    count = 0
    question = ''
    for que in questions:
        question = que.lower().translate(str.maketrans('', '', string.punctuation))

    for i in list(answer.strip().split(' ')):
      if i not in  ['a', 'is','by', 'was', 'which', 'man', 'did', 'do', 'woman', 'boy', 'girl', 'person','people','scene','frame','in','to','the', 'of', 'on','with', 'from', 'at', 'and']:
          if i in words:
            index_list.append(main[main['question_id']==q].index[0])

print(index_list)
#QA.drop(index_list, inplace = True)
main = main[main.index.isin(index_list)]
main.info()


[3, 6, 8, 9, 13, 17, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 33, 35, 36, 37, 41, 42, 43, 44, 45, 49, 50, 51, 52, 53, 54, 56, 57, 58, 60, 61, 64, 65, 66, 75, 76, 80, 81, 82, 85, 86, 90, 93, 94, 96, 97, 98, 99, 102, 104, 106, 107, 108, 109, 110, 111, 113, 114, 115, 116, 121, 128, 129, 131, 140, 141, 143, 144, 145, 147, 148, 149, 150, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 175, 176, 177, 178, 179, 180, 182, 184, 185, 187, 189, 190, 192, 195, 199, 200, 201, 202, 203, 204, 205, 206, 209, 210, 212, 217, 219, 220, 221, 222, 223, 229, 232, 236, 238, 239, 240, 243, 244, 245, 246, 247, 248, 254, 255, 256, 257, 258, 259, 261, 265, 266, 267, 269, 271, 272, 274, 277, 278, 290, 291, 292, 296, 297, 298, 303, 307, 308, 311, 312, 313, 318, 319, 321, 323, 326, 327, 328, 329, 330, 339, 345, 349, 350, 351, 352, 361, 363, 364, 367, 372, 374, 375, 376, 377, 379, 380, 382, 383, 386, 387, 390, 391, 394, 395, 400, 401, 402, 403, 405, 406, 407, 408, 41

In [4]:
import ffmpeg

def segment_video(input_path, output_path, start, end):
    input_file = ffmpeg.input(directory_path+'/'+input_path+'.mp4')
    duration = int(end-start)
    if duration < 10:
      output_file = ffmpeg.output(input_file.trim(start=start, duration = duration).filter('setpts','PTS-STARTPTS'), directory_path+'/trim/'+output_path+'.mp4')
      ffmpeg.run(output_file)
      return True
    else:
      return False

In [5]:


def run_oic(question_id, video_id, question, choices, start,end):
    out_dir = "out"

    # check if the directory has a video file
    for video in os.scandir(directory_path):
        if video.is_file():
            if os.path.splitext(video.name)[0] == video_id:
                #segment the video into frames with the given start and end of the video
                video_p = segment_video(os.path.splitext(video.name)[0],question_id+os.path.splitext(video.name)[0],start,end)
                # if video exists, extract frames with windowing of few frames to improve the detection
                if video_p:
                  ex = FrameExtractor(video_file=directory_path+'/trim/'+question_id+video.name,fps_to_save=10, window_size=3)
                  ex.main()
                  
                  # instantiate graphene "OIC core that runs RelTR"  
                  g = graphene.Graphene(alpha=0.3, min_assignment_conf=0.6)
                
                  # check if the out directory exists else create one
                  if not os.path.isdir(out_dir):
                      os.mkdir(out_dir)
                  
                  # prepare output files 
                  text = question_id+os.path.splitext(video.name)[0]+'graph2text.txt'
                  img_dir_path = directory_path+'/trim/'+question_id+os.path.splitext(video.name)[0]+'-opencv'  
                  
                  # classify images from the image directory
                  g.classify_images_window(img_dir_path,5)
                  
                  # generate relationship graph
                  graph_dir_path = img_dir_path + "/img/JSON"
                  g.generate_temporal_graph_frames_no_plot(scenegraphs_path=graph_dir_path, image_path=img_dir_path + "/img")

                  # save textual output in the out directory
                  g.tg.to_text(os.path.join(out_dir, text))
      
                  if os.path.isfile(os.path.join(out_dir,text)):
                      with open(os.path.join(out_dir,text)) as f:
                          context = "".join(map(str,f.readlines()))
                          return str(context)
                  else:
                    continue

In [6]:
import shutil
shutil.rmtree('temp')

In [7]:
#import cProfile

q_ids = main['question_id'].unique()
print(len(q_ids))
for q in q_ids:
  #if 'Seq' in q:
    que = main.query("question_id == '"+q+"'")
    video_id = que['video_id'].values[0]
    question = que['question'].values[0]
    answer = que['answer'].values[0].lower().strip('the ').translate(str.maketrans('', '', string.punctuation))
    start = que['start'].values[0]
    end = que['end'].values[0]
    choices = dict()
    choice_string = ''
    options = main.query("question_id == '"+q+"'")["choices"].values[0]
    print(options)
    for choice in options:
          choices.update({choice['choice_id']:choice['choice'].lower().strip('the').translate(str.maketrans('', '', string.punctuation)).strip()})
          choice_string += ' ('+str(choice['choice_id']+1)+')'+str(choice['choice'].lower().strip('the').translate(str.maketrans('', '', string.punctuation)))

    if not os.path.isdir(directory_path+'/trim/'+q+video_id+'-opencv'):
      #cProfile.run('run_oic(q, video_id, question, choices, start, end)')
      prompt = run_oic(q, video_id, question, choices, start, end)
      if prompt is not None:
        main.loc[main['question_id'] == q, 'OIC_context'] = prompt
        formatted_question = question+' Guess the most likely answer among these four options: '+choice_string+' Respond only with a single number between 1 and 4. Do not produce any other output.'
        response = run_gpt(prompt, formatted_question)
        main.loc[main['question_id'] == q, 'OIC_answer'] = str(response)
        main.loc[main['question_id'] == q, 'OIC_question'] = formatted_question
        OIC_answer = response
        print(choices)
        print (int(OIC_answer))
        print(choices[int(OIC_answer)-1])
        print(answer)
        if choices[int(OIC_answer)-1] == answer:
          main.loc[main['question_id'] == q, 'Match'] = 'Correct'
          print('correct')
        else:
          print('wrong')
          main.loc[main['question_id'] == q, 'Match'] = 'Wrong'
        print('-'*100)
        print('OIC question: {}'.format(formatted_question))
        print('OIC answer: {}'.format(choices[int(OIC_answer)-1]))
        main.to_csv('STAR_eval/datatables5/question_answers.csv')
          
main.head()
main.name = 'all data'

2892
[{'choice_id': 0, 'choice': 'The food.', 'choice_program': [{'function': 'Equal', 'value_input': ['food']}]}, {'choice_id': 1, 'choice': 'The shoe.', 'choice_program': [{'function': 'Equal', 'value_input': ['shoe']}]}, {'choice_id': 2, 'choice': 'The blanket.', 'choice_program': [{'function': 'Equal', 'value_input': ['blanket']}]}, {'choice_id': 3, 'choice': 'The sandwich.', 'choice_program': [{'function': 'Equal', 'value_input': ['sandwich']}]}]
[{'choice_id': 0, 'choice': 'The food.', 'choice_program': [{'function': 'Equal', 'value_input': ['food']}]}, {'choice_id': 1, 'choice': 'The shoe.', 'choice_program': [{'function': 'Equal', 'value_input': ['shoe']}]}, {'choice_id': 2, 'choice': 'The pillow.', 'choice_program': [{'function': 'Equal', 'value_input': ['pillow']}]}, {'choice_id': 3, 'choice': 'The towel.', 'choice_program': [{'function': 'Equal', 'value_input': ['towel']}]}]


ffmpeg version 6.0 Copyright (c) 2000-2023 the FFmpeg developers
  built with Apple clang version 15.0.0 (clang-1500.0.40.1)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/6.0_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --enable-libsoxr --enable-libzmq --enabl

STAR_eval/Charades/trim/Interaction_T1_704GLAP.mp4
6.134
2.0
4.075643951744375
9


100%|██████████| 24/24 [01:50<00:00,  4.59s/it]


copying images and cleaning up temporary files:


100%|██████████| 24/24 [00:00<00:00, 3055.03it/s]


STAR_eval/Charades/trim/Interaction_T1_704GLAP-opencv/frame0-00-00.33.jpg STAR_eval/Charades/trim/Interaction_T1_704GLAP-opencv/img/frame0-00-00.33.jpg
STAR_eval/Charades/trim/Interaction_T1_704GLAP-opencv/frame0-00-01.30.jpg STAR_eval/Charades/trim/Interaction_T1_704GLAP-opencv/img/frame0-00-01.30.jpg
STAR_eval/Charades/trim/Interaction_T1_704GLAP-opencv/frame0-00-02.12.jpg STAR_eval/Charades/trim/Interaction_T1_704GLAP-opencv/img/frame0-00-02.12.jpg
STAR_eval/Charades/trim/Interaction_T1_704GLAP-opencv/frame0-00-02.93.jpg STAR_eval/Charades/trim/Interaction_T1_704GLAP-opencv/img/frame0-00-02.93.jpg
STAR_eval/Charades/trim/Interaction_T1_704GLAP-opencv/frame0-00-03.75.jpg STAR_eval/Charades/trim/Interaction_T1_704GLAP-opencv/img/frame0-00-03.75.jpg
{0: 'food', 1: 'shoe', 2: 'pillow', 3: 'towel'}


ValueError: invalid literal for int() with base 10: 'The text does not provide information about any object being thrown by the person.'